In [2]:
# ============================================================
# STEP 3 OF RAG: RETRIEVING RELEVANT CONTEXT
# Google Colab - Complete Corrected Code
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL REQUIRED LIBRARIES
# ------------------------------------------------------------

!pip install -q -U langchain-text-splitters chromadb sentence-transformers


# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions


# ------------------------------------------------------------
# 3. CREATE SAMPLE DOCUMENTS
# ------------------------------------------------------------

documents = {

    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
""",

    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
"""
}


# ------------------------------------------------------------
# 4. SPLIT DOCUMENTS INTO CHUNKS
# ------------------------------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("=" * 60)
print("DOCUMENT CHUNKING")
print("=" * 60)

all_chunks = {}

for source, text in documents.items():

    chunks = splitter.split_text(text)
    all_chunks[source] = chunks

    print(f"\nSource: {source}")
    print(f"Number of chunks: {len(chunks)}")

    for i, chunk in enumerate(chunks):
        print(f"\nChunk {i}:")
        print(chunk)


# ------------------------------------------------------------
# 5. CREATE CHROMADB CLIENT
# ------------------------------------------------------------

client = chromadb.Client()


# ------------------------------------------------------------
# 6. LOAD HUGGING FACE EMBEDDING MODEL
# ------------------------------------------------------------

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

print("\n" + "=" * 60)
print("EMBEDDING MODEL")
print("=" * 60)

print("Embedding model loaded: all-MiniLM-L6-v2")


# ------------------------------------------------------------
# 7. CREATE OR GET CHROMADB COLLECTION
# ------------------------------------------------------------

# get_or_create_collection() prevents an error if this cell
# is executed multiple times in the same Colab session.

collection = client.get_or_create_collection(
    name="rag_chunks",
    embedding_function=hf_embedder,
    configuration={"hnsw": {"space": "cosine"}}
)

print("\nCollection ready:", collection.name)


# ------------------------------------------------------------
# 8. STORE CHUNKS IN CHROMADB
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STORING CHUNKS")
print("=" * 60)

for source, chunks in all_chunks.items():

    ids = [
        f"{source}_chunk_{i}"
        for i in range(len(chunks))
    ]

    metadatas = [
        {
            "source": source,
            "chunk_index": i,
            "char_count": len(chunk)
        }
        for i, chunk in enumerate(chunks)
    ]

    # Check whether these IDs already exist.
    existing = collection.get(
        ids=ids,
        include=["metadatas"]
    )

    existing_ids = set(existing["ids"])

    new_chunks = []
    new_metadatas = []
    new_ids = []

    for chunk, metadata, chunk_id in zip(
        chunks,
        metadatas,
        ids
    ):

        if chunk_id not in existing_ids:

            new_chunks.append(chunk)
            new_metadatas.append(metadata)
            new_ids.append(chunk_id)

    if new_ids:

        collection.add(
            documents=new_chunks,
            metadatas=new_metadatas,
            ids=new_ids
        )

        print(
            f"{source}: {len(new_ids)} new chunks added."
        )

    else:

        print(
            f"{source}: chunks already exist."
        )


print("\nTotal chunks stored:", collection.count())


# ------------------------------------------------------------
# 9. VIEW STORED CHUNKS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STORED CHUNKS")
print("=" * 60)

stored = collection.get(
    include=["documents", "metadatas"]
)

for chunk_id, document, metadata in zip(
    stored["ids"],
    stored["documents"],
    stored["metadatas"]
):

    print(f"\nID: {chunk_id}")
    print(f"Source: {metadata['source']}")
    print(f"Chunk index: {metadata['chunk_index']}")
    print(f"Text: {document[:120]}...")
    print("-" * 60)


# ------------------------------------------------------------
# 10. BASIC RETRIEVAL
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("1. BASIC RETRIEVAL")
print("=" * 60)

user_query = "How does RAG improve answers from a language model?"

results = collection.query(
    query_texts=[user_query],
    n_results=3,
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

print("\nQuery:", user_query)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):

    print(
        f"\nDistance: {distance:.4f}"
        f"\nSource: {meta['source']}"
        f"\nChunk: {meta['chunk_index']}"
    )

    print("\nText:")
    print(doc)

    print("-" * 60)


# ------------------------------------------------------------
# 11. DISTANCE TO SIMILARITY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("2. DISTANCE VS SIMILARITY")
print("=" * 60)


def distance_to_similarity(distance):
    """
    ChromaDB is configured to use cosine distance.

    cosine similarity = 1 - cosine distance
    """

    return 1 - distance


for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):

    similarity = distance_to_similarity(distance)

    print(
        f"Similarity: {similarity:.4f}"
        f" | Source: {meta['source']}"
        f" | Chunk: {meta['chunk_index']}"
    )


# ------------------------------------------------------------
# 12. RETRIEVAL WITH SIMILARITY THRESHOLD
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("3. SIMILARITY THRESHOLD")
print("=" * 60)


def retrieve_with_threshold(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.30
):

    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    filtered = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):

        similarity = distance_to_similarity(distance)

        if similarity >= similarity_threshold:

            filtered.append({
                "text": doc,
                "metadata": meta,
                "similarity": similarity
            })

    return filtered


# ------------------------------------------------------------
# 13. TEST RELEVANT QUERY
# ------------------------------------------------------------

relevant_query = "What is deep learning?"

relevant_matches = retrieve_with_threshold(
    relevant_query,
    collection,
    n_results=5,
    similarity_threshold=0.30
)

print("\nRelevant query:", relevant_query)

print(
    f"Chunks passing threshold: "
    f"{len(relevant_matches)}"
)

for match in relevant_matches:

    print(
        f"\nSimilarity: "
        f"{match['similarity']:.4f}"
    )

    print(
        f"Source: "
        f"{match['metadata']['source']}"
    )

    print(
        f"Text: "
        f"{match['text'][:150]}..."
    )


# ------------------------------------------------------------
# 14. TEST UNRELATED QUERY
# ------------------------------------------------------------

unrelated_query = "What is the best recipe for chocolate cake?"

unrelated_matches = retrieve_with_threshold(
    unrelated_query,
    collection,
    n_results=5,
    similarity_threshold=0.30
)

print("\n" + "-" * 60)

print("Unrelated query:", unrelated_query)

print(
    f"Chunks passing threshold: "
    f"{len(unrelated_matches)}"
)

if len(unrelated_matches) == 0:

    print(
        "No relevant context found."
    )

    print(
        "A real RAG system should avoid guessing "
        "when the knowledge base has no useful information."
    )


# ------------------------------------------------------------
# 15. SEMANTIC SEARCH + METADATA FILTER
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("4. SEMANTIC SEARCH + METADATA FILTER")
print("=" * 60)

filtered_query = (
    "How does search work with meaning instead of keywords?"
)

results = collection.query(
    query_texts=[filtered_query],
    n_results=2,

    where={
        "source": "embeddings_in_practice.txt"
    },

    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

print("\nQuery:", filtered_query)
print("Source filter: embeddings_in_practice.txt")

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):

    similarity = distance_to_similarity(distance)

    print(
        f"\nSimilarity: {similarity:.4f}"
    )

    print(
        f"Source: {meta['source']}"
    )

    print("\nText:")
    print(doc)

    print("-" * 60)


# ------------------------------------------------------------
# 16. REUSABLE RETRIEVE_CONTEXT FUNCTION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("5. REUSABLE RETRIEVAL FUNCTION")
print("=" * 60)


def retrieve_context(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.25,
    source_filter=None
):

    query_kwargs = {
        "query_texts": [query],
        "n_results": n_results,
        "include": [
            "documents",
            "metadatas",
            "distances"
        ]
    }

    # Add metadata filter only when requested.
    if source_filter is not None:

        query_kwargs["where"] = {
            "source": source_filter
        }

    results = collection.query(
        **query_kwargs
    )

    context_chunks = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):

        similarity = distance_to_similarity(
            distance
        )

        if similarity >= similarity_threshold:

            context_chunks.append({

                "text": doc,

                "source": meta["source"],

                "chunk_index": meta["chunk_index"],

                "similarity": round(
                    similarity,
                    4
                )
            })

    return context_chunks


# ------------------------------------------------------------
# 17. TEST retrieve_context()
# ------------------------------------------------------------

context = retrieve_context(
    "What is machine learning?",
    collection,
    n_results=5,
    similarity_threshold=0.25
)

print("\nRetrieved context:")

if context:

    for item in context:

        print(
            f"\nSimilarity: "
            f"{item['similarity']}"
        )

        print(
            f"Source: "
            f"{item['source']}"
        )

        print(
            f"Chunk: "
            f"{item['chunk_index']}"
        )

        print(
            f"Text: "
            f"{item['text']}"
        )

else:

    print(
        "No relevant context found."
    )


# ------------------------------------------------------------
# 18. FORMAT RETRIEVED CONTEXT FOR LLM
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("6. LLM-READY PROMPT")
print("=" * 60)


def format_context_for_llm(
    query,
    context_chunks
):

    if not context_chunks:

        return (
            f"Question: {query}\n\n"
            "No relevant context was found "
            "in the knowledge base."
        )

    context_text = "\n\n".join(

        f"[Source: {item['source']}, "
        f"chunk {item['chunk_index']}]\n"
        f"{item['text']}"

        for item in context_chunks
    )

    prompt = f"""Answer the question using ONLY the context below.

If the answer is not available in the context, say:
"I don't know based on the available information."

Context:

{context_text}

Question:
{query}

Answer:"""

    return prompt


# ------------------------------------------------------------
# 19. CREATE FINAL PROMPT
# ------------------------------------------------------------

final_query = (
    "How does RAG combine search with a language model?"
)

context = retrieve_context(
    final_query,
    collection,
    n_results=5,
    similarity_threshold=0.25
)

final_prompt = format_context_for_llm(
    final_query,
    context
)

print(final_prompt)


# ------------------------------------------------------------
# 20. TEST WITH PARAPHRASED QUESTIONS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("7. PARAPHRASED QUERY TESTING")
print("=" * 60)

test_queries = [

    "Why do companies use vector databases?",

    "What's the difference between AI and machine learning?",

    "How do chatbots understand differently worded questions?"

]


for query in test_queries:

    print("\n" + "-" * 60)

    print("Query:", query)

    matches = retrieve_context(
        query,
        collection,
        n_results=2,
        similarity_threshold=0.25
    )

    if not matches:

        print(
            "No relevant chunks found."
        )

        continue

    for match in matches:

        print(
            f"\nSimilarity: "
            f"{match['similarity']}"
        )

        print(
            f"Source: "
            f"{match['source']}"
        )

        print(
            f"Text: "
            f"{match['text'][:200]}..."
        )


# ------------------------------------------------------------
# 21. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 3 RAG RETRIEVAL COMPLETED SUCCESSFULLY")
print("=" * 60)

print("\nConcepts covered:")

print("1. Document chunking")
print("2. ChromaDB vector storage")
print("3. Basic semantic retrieval")
print("4. n_results / top-k retrieval")
print("5. Cosine distance")
print("6. Similarity score")
print("7. Similarity threshold")
print("8. Metadata filtering")
print("9. Reusable retrieve_context()")
print("10. LLM-ready prompt formatting")
print("11. Paraphrased query testing")

print("\nTotal chunks stored:", collection.count())
print("Embedding model: all-MiniLM-L6-v2")
print("Vector database: ChromaDB")

print("\nSTEP 3 COMPLETE!")
print("Next step: Pass the retrieved context to an LLM.")

DOCUMENT CHUNKING

Source: ai_overview.txt
Number of chunks: 8

Chunk 0:
Artificial Intelligence: An Overview

Chunk 1:
Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include

Chunk 2:
understanding language, recognizing images, making decisions, and learning from experience.

Chunk 3:
Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Chunk 4:
Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as

Chunk 5:
recognizing faces in photos or understanding the meaning of a sentence.

Chunk 6:
Retrieval-Augmented Generation, or RAG, is a technique t